In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 — Extração de Features
# MAGIC Fonte: `default.gold_indicador_municipio`
# MAGIC Target: `status_meta` (2024) — binária

# COMMAND ----------

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
TABELA = f"{CATALOGO}.default.gold_indicador_municipio"

df = spark.table(TABELA)
print(f"Total de registros: {df.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Separar base histórica (2023) e base de modelagem (2024)

# COMMAND ----------

# 2023: apenas a taxa observada, usada como feature histórica
hist = (
    df.filter("ano = 2023")
      .select(
          F.col("id_municipio"),
          F.col("taxa_observada").alias("taxa_2023"),
          F.col("media_portugues").alias("media_portugues_2023"),
          F.col("percentual_participacao").alias("participacao_2023"),
      )
)
print(f"Municípios com histórico 2023: {hist.count()}")

# COMMAND ----------

# 2024: base de modelagem — só quem tem target
base = (
    df.filter("ano = 2024 AND status_meta IN ('abaixo', 'atingiu')")
      .select(
          "id_municipio",
          "qtd_redes",
          "percentual_participacao",
          "meta_do_ano",
          "meta_2025", "meta_2026", "meta_2027",
          "meta_2028", "meta_2029", "meta_2030",
          "tem_dado_completo",
          "flag_outlier",
          F.when(F.col("status_meta") == "atingiu", 1).otherwise(0).alias("target"),
      )
)
print(f"Municípios com target em 2024: {base.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Join com histórico e criação de features derivadas

# COMMAND ----------

modelo = (
    base.join(hist, on="id_municipio", how="left")
        .withColumn("tem_historico", F.col("taxa_2023").isNotNull().cast("int"))
        .withColumn("gap_meta_2023",
                    F.when(F.col("taxa_2023").isNotNull(),
                           F.col("taxa_2023") - F.col("meta_do_ano")))
        .withColumn("esforco_necessario",
                    F.when(F.col("taxa_2023").isNotNull(),
                           F.col("meta_do_ano") - F.col("taxa_2023")))
)

print(f"Base final: {modelo.count()} registros")
modelo.groupBy("target").count().show()
modelo.select("tem_historico").groupBy("tem_historico").count().show()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Persistir como tabela Delta para as próximas etapas

# COMMAND ----------

(modelo.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOGO}.default.fase3_features_municipio"))

print("Tabela criada: default.fase3_features_municipio")
spark.table(f"{CATALOGO}.default.fase3_features_municipio").printSchema()